In [57]:
import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree
from sklearn.model_selection import cross_val_score

from typing import Annotated
from abc import ABC, abstractmethod
from dataclasses import dataclass
import re

from geneticengine.grammar.decorators import abstract
from geneticengine.grammar.grammar import extract_grammar
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.random.sources import NativeRandomSource
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.problems import SingleObjectiveProblem
from geneticengine.evaluation.budget import EvaluationBudget, TimeBudget
from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.problems import MultiObjectiveProblem
from geneticengine.algorithms.gp.operators.combinators import SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection

import time


In [ ]:
df = pd.read_csv('../../datasets/synthetic_classification_data.csv')
# df = pd.read_csv('../../datasets/dataset_gp.csv')
X = df.drop(columns=['target'])
y = df['target']
n_features = X.shape[1]
feature_names = X.columns

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_values = X_train.values
X_test_values = X_test.values

In [80]:
baseline_model = tree.DecisionTreeClassifier(max_depth=5, random_state=42)
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
baseline_scores = cross_val_score(baseline_model, X_train, y_train, cv=cv_strategy, scoring='accuracy')
print(f"Baseline CV Accuracy on original features: {np.mean(baseline_scores)}")

Baseline CV Accuracy on original features: 0.9249999999999999


In [61]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
importances = rf.feature_importances_
n_top_features = int(n_features * 0.3)
n_top_features = max(1, n_top_features)
top_indices = np.argsort(importances)[::-1][:n_top_features]
top_feature_names = [feature_names[i] for i in top_indices]

In [62]:
top_feature_names

['f3', 'f2', 'f7']

In [63]:
class Feature(ABC):
    @abstractmethod
    def evaluate(self, X):pass
    
    @abstractmethod
    def count_operators(self, X):pass

    @abstractmethod
    def get_unique_columns(self, unique_indices: set):pass

    @abstractmethod
    def get_depth(self) -> int:pass
    

@dataclass
class PrimitiveFeatures(Feature):
    top_feature_idx: Annotated[int, IntRange(0, len(top_indices)-1)]
    def evaluate(self, X):
        column_index = top_indices[self.top_feature_idx]
        return X[:, column_index]
    def __str__(self):
        column_index = top_indices[self.top_feature_idx]
        name = feature_names[column_index]
        safe = re.sub(r'[^A-Za-z0-9_]+', '_', str(name))
        return f"{safe}"
    def count_operators(self) -> int:
        return 0 #there is no operators in features
    def get_unique_columns(self, unique_indices:set):
        unique_indices.add(self.top_feature_idx) #add it index to the unique_indices set
    def get_depth(self) -> int:
        return 1
    
@dataclass
class Add(Feature):
    left : Feature #type feature
    right : Feature #type feature
    def evaluate(self, X):
        return self.left.evaluate(X) + self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} + {self.right})"
    def count_operators(self)->int:
        return 1 + self.left.count_operators() + self.right.count_operators() #add 1 operator and the count of operators bellow
    def get_unique_columns(self, unique_indices:set):
        self.left.get_unique_columns(unique_indices)
        self.right.get_unique_columns(unique_indices)
    def get_depth(self)-> int:
        return 1 + max(self.left.get_depth(), self.right.get_depth()) # add 1 depth and the max depth of its childs

@dataclass
class Subtract(Feature):
    left : Feature
    right : Feature
    def evaluate(self, X):
        return self.left.evaluate(X) - self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} - {self.right})"
    def count_operators(self)->int:
        return 1 + self.left.count_operators() + self.right.count_operators() 
    def get_unique_columns(self, unique_indices:set):
        self.left.get_unique_columns(unique_indices)
        self.right.get_unique_columns(unique_indices)
    def get_depth(self)-> int:
        return 1 + max(self.left.get_depth(), self.right.get_depth())

@dataclass
class Multiply(Feature):
    left: Feature
    right: Feature
    def evaluate(self, X):
        return self.left.evaluate(X) * self.right.evaluate(X)
    def __str__(self):
        return f"({self.left} * {self.right})"
    def count_operators(self)->int:
        return 1 + self.left.count_operators() + self.right.count_operators() 
    def get_unique_columns(self, unique_indices:set):
        self.left.get_unique_columns(unique_indices)
        self.right.get_unique_columns(unique_indices)
    def get_depth(self)-> int:
        return 1 + max(self.left.get_depth(), self.right.get_depth())



components = [
    PrimitiveFeatures,
    Add,
    Subtract,
    Multiply
]

grammar = extract_grammar(components, Feature)
print(grammar)

Grammar<Starting=Feature,Productions={
Feature -> PrimitiveFeatures(top_feature_idx: Annotated[int])|
	Add(left: Feature, right: Feature)|
	Subtract(left: Feature, right: Feature)|
	Multiply(left: Feature, right: Feature)
}


In [64]:
def fitness_function(feature: Feature) -> list[float]:
    start_time = time.time()

    new_feature_values = feature.evaluate(X_train_values).reshape(-1,1)

    X_combined = np.hstack([X_train_values, new_feature_values])

    clf = make_pipeline(tree.DecisionTreeClassifier(random_state=42))

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1 = cross_val_score(clf, X_combined, y_train, cv=cv, scoring='f1_macro').mean()

    operators = feature.count_operators()
    unique_indices = set()
    feature.get_unique_columns(unique_indices)
    unique_cols = len(unique_indices)
    depth = feature.get_depth()

    execution_time = time.time() - start_time

    return [f1, execution_time, operators, unique_cols, depth]

In [65]:
step = SequenceStep(
    LexicaseSelection(True),
    GenericCrossoverStep(0.01),
    GenericMutationStep(0.9)
)

In [66]:
# #GP setup
rnd = NativeRandomSource(123)
decider = MaxDepthDecider(rnd, grammar, max_depth=5)
representation = TreeBasedRepresentation(grammar, decider)
objective = MultiObjectiveProblem(fitness_function=fitness_function, minimize=[False, True, True, True, True])

# gp = GeneticProgramming(
#     problem=objective,
#     budget=TimeBudget(30),
#     representation=representation,
#     random=rnd,
#     step=step,
#     tracker= ProgressTracker(
#         objective,
#         recorders=[CSVSearchRecorder(csv_path='../../gp_outputs/tests.csv', problem=objective, fields={"Eval Time": lambda t,i,p:i.get_fitness(p).fitness_components[1],
#                                                                                                        "F1 Score": lambda t, i, p: i.get_fitness(p).fitness_components[0],
#                                                                                                        "Expression": lambda t, i, p: i.get_phenotype(),
#                                                                                                        "Operators": lambda t, i, p: i.get_fitness(p).fitness_components[2],
#                                                                                                        "Unique Columns": lambda t, i, p: i.get_fitness(p).fitness_components[3],
#                                                                                                        "Depth": lambda t, i, p: i.get_fitness(p).fitness_components[4],
                                                                                                       
#                                                                                                        }, only_record_best_individuals=True)]
#     ),
#     population_size=50,
# )


In [67]:
# solutions = gp.search()

In [68]:
# best_f1 = max(solutions, key=lambda row: row.get_fitness(objective).fitness_components[0])
# print(best_f1.get_fitness(objective).fitness_components, best_f1.get_phenotype())

In [69]:
decider = MaxDepthDecider(rnd, grammar, max_depth=5)
representation = TreeBasedRepresentation(grammar, decider)
objective = MultiObjectiveProblem(fitness_function=fitness_function, minimize=[False, True, True, True, True])

candidate_pool = []
for i in range(10):
    rnd = NativeRandomSource(123+i)
    gp = GeneticProgramming(
        problem=objective,
        budget=TimeBudget(10),
        representation=representation,
        random=rnd,
        step=step,
        tracker= ProgressTracker(
            objective,
        ),
        population_size=50,
    )
    solutions = gp.search()
    best_f1 = max(solutions, key=lambda row: row.get_fitness(objective).fitness_components[0])

    candidate_pool.append(best_f1.get_phenotype())
    print(f"Run {i+1}: {best_f1.get_phenotype()}")
    fitness_value = best_f1.get_fitness(objective).fitness_components[0]
    print(f"Best F1 Score: {fitness_value}")
    print(f"Execution time: {best_f1.get_fitness(objective).fitness_components[1]}")
    



Run 1: (f2 - (((f2 - f3) + (f3 * f2)) - ((f2 + f7) - f3)))
Best F1 Score: 0.9346839137548011
Execution time: 0.06040048599243164
Run 2: (((f2 - (f2 - f3)) + f7) + (f2 - ((f7 - f7) * f3)))
Best F1 Score: 0.942009749660264
Execution time: 0.056012868881225586
Run 3: (f7 + f2)
Best F1 Score: 0.9384782817176566
Execution time: 0.058267831802368164
Run 4: (f2 + (((f3 + f3) + (f7 - f3)) - f3))
Best F1 Score: 0.9384782817176566
Execution time: 0.058012962341308594
Run 5: (f7 + f2)
Best F1 Score: 0.9384782817176566
Execution time: 0.0584406852722168
Run 6: (f7 + f2)
Best F1 Score: 0.9384782817176566
Execution time: 0.0577397346496582
Run 7: ((((f2 - f7) + (f7 * f7)) - f2) - (((f3 + f2) + f7) + ((f3 * f2) + f2)))
Best F1 Score: 0.9378094757526299
Execution time: 0.05801510810852051
Run 8: (f2 + f7)
Best F1 Score: 0.9384782817176566
Execution time: 0.06511521339416504
Run 9: (f7 + f2)
Best F1 Score: 0.9384782817176566
Execution time: 0.06031632423400879
Run 10: (f7 + f2)
Best F1 Score: 0.9384782

In [70]:
unique_features = list({str(f): f for f in candidate_pool}.values())
new_features_df = pd.DataFrame()

for i, feature in enumerate(unique_features):
    col_name = f"{feature}"
    new_column_values = feature.evaluate(X.values)
    new_features_df[col_name] = new_column_values

final_df = pd.concat([df, new_features_df], axis=1)

final_df.to_csv('../../datasets/dataset_gp.csv')
